# Quiet-period HR / pulse period versus PPG amplitude

Polar Verity Sense analysis using the existing recording loader, movement-burst
detector, pulse-fiducial design, and interval artifact classifier. **This notebook
uses the whole recording; sleep is unverified.** Accelerometer quietness does not
establish sleep. No sleep model, Garmin synchronization, or manual clock shift runs here.

**Reference:** [Silvani et al. (2008), Sleep-dependent changes in the coupling
between heart period and blood pressure in human subjects](https://doi.org/10.1152/ajpregu.00756.2007).
The [author-posted full text](https://www.researchgate.net/publication/5578779_Sleep-dependent_changes_in_the_coupling_between_heart_period_and_blood_pressure_in_human_subjects)
was checked against the Methods and Fig. 2.

The paper analyzes ECG heart period and systolic blood pressure in scored sleep
states. It uses 5-minute subsets at 1-minute steps, linear interpolation at 4 Hz,
lags ±25 s, zero-lag autocorrelation normalization, and a second analysis below
0.15 Hz with a 10-pole Butterworth filter. Its curves are averaged within subjects
before group summaries. **Here PWA replaces pressure and PPG pulse period replaces
ECG heart period.** These signals are not interchangeable physiological measures.
No CCF pattern here establishes causality or baroreflex function.

## Setup and configuration

Run from the repository root or `offline_processing`. Install
`offline_processing/requirements-notebooks.txt` in the selected Python kernel.
Set `verity_sense_path` through the existing ignored `paths.local.json` or
`OVERNIGHT_VERITY_SENSE_PATH`; no personal path is embedded in this notebook.

Defaults retain the requested method. Buffers expand each burst from **2 s before
onset through 5 s after offset**. Only the intervals between consecutive bursts
are candidates; recording edges are excluded, including a recording with fewer
than two bursts. A 0.020 g movement-envelope threshold is inherited from the
existing wrist analysis; its transfer to Polar's wear site is exploratory and
must be inspected in the overview. All burst durations are used.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import importlib.metadata
import hashlib
import json
import platform
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROCESSING = Path.cwd() if (Path.cwd() / "notebook_paths.py").exists() else Path.cwd() / "offline_processing"
PROCESSING = PROCESSING.resolve()
if not (PROCESSING / "notebook_paths.py").exists():
    raise FileNotFoundError("Run from the repository root or offline_processing")
if str(PROCESSING) not in sys.path:
    sys.path.insert(0, str(PROCESSING))

from notebook_paths import input_path
from polar_interpretability import load_polar_recording
from detect_acc_bursts import detect_bursts_in_intervals
from analysis_windows import intersect_intervals, validate_intervals
from polar_ccf import (
    CCFConfig, analyze_quiet_ccf, availability, extract_beats, extrema,
    plot_ccf, plot_extrema, plot_overview, plot_representative,
    quiet_periods, representative_signals, sensor_clock_index,
)

CONFIG = CCFConfig()  # 300 s, 60 s step, 4 Hz, ±25 s, order 10, 0.15 Hz
PPG_FS = 55.0         # waveform regularization, not CCF resampling
ACC_FS = 52.0         # existing polar.ipynb setting; timestamps drive regularization
ACC_THRESHOLD_G = 0.020
ACC_INPUT_UNIT = "mg"
MAX_ACC_GAP_S = 0.25
MAX_PPG_GAP_S = 0.10
PPG_POLARITY = -1    # existing polar.ipynb sign reversal
SENSOR_CLOCK_ZONE = "UTC"  # bookkeeping label; not verified civil time
MIN_WINDOWS_TO_SUMMARIZE = 2  # reporting threshold; never changes window geometry
REPRESENTATIVE_WINDOW_ID = None  # None selects by quality, not correlation strength
MAX_HIGHLIGHTED_WINDOWS = 3
OUTPUT_ROOT = PROCESSING / "outputs" / "polar_quiet_ccf"
plt.style.use("seaborn-v0_8-whitegrid")
display(pd.Series(asdict(CONFIG), name="CCF configuration"))

## Load and check alignment

The existing loader reads PPG channel 0 and XYZ acceleration, converting the
shared Polar nanosecond counter from the 2000 epoch. Sensor clock labels are
preserved: both streams receive the same declared timezone and neither is moved
to a Garmin clock. Missing, duplicate, unsorted, nonfinite samples, or long raw
gaps fail explicitly. The alignment report verifies overlapping acquisition
support; it cannot independently establish hardware synchronization.

The waveform is regularized at 55 Hz before applying the existing 0.5–8 Hz,
order-3, sign-reversed preprocessing. The reusable `ppg_fiducials.py` carries the
notebook's Aboy++-style detector with necessary implementation corrections
(digital Butterworth SOS filtering with passband cutoff semantics, actual onset sample indices, final-window coverage,
centered derivative, removal of an inconsistent BPM/sample comparison). This
adapted detector needs waveform inspection; it is not an exact pyPPG release.

In [ ]:
raw_ppg, raw_acc = load_polar_recording(input_path("verity_sense_path"))
raw_ppg = raw_ppg.copy()
raw_acc = raw_acc.copy()
raw_ppg.index = sensor_clock_index(raw_ppg.index, SENSOR_CLOCK_ZONE)
raw_acc.index = sensor_clock_index(raw_acc.index, SENSOR_CLOCK_ZONE)

alignment_rows = []
for label, signal in (("PPG", raw_ppg), ("ACC", raw_acc)):
    intervals_s = np.diff(signal.index.asi8) / 1e9
    if not np.isfinite(signal.to_numpy()).all():
        raise ValueError(f"Nonfinite {label} samples")
    alignment_rows.append({"stream": label, "start": signal.index[0], "last_sample": signal.index[-1],
                           "samples": len(signal), "observed_median_hz": 1 / np.median(intervals_s),
                           "max_gap_s": intervals_s.max()})
alignment = pd.DataFrame(alignment_rows).set_index("stream")
common_start = max(raw_ppg.index[0], raw_acc.index[0])
common_end = min(raw_ppg.index[-1], raw_acc.index[-1])  # conservative final sample bound
if common_end <= common_start:
    raise ValueError("PPG and ACC do not overlap; inspect sensor clocks")
common_s = (common_end - common_start).total_seconds()
shorter_s = min((x.index[-1] - x.index[0]).total_seconds() for x in (raw_ppg, raw_acc))
if common_s / shorter_s < .95:
    raise ValueError("Less than 95% common support; inspect clocks/acquisition before proceeding")
if alignment.loc["ACC", "max_gap_s"] > MAX_ACC_GAP_S:
    raise ValueError("Split ACC into continuous runs before movement detection")
if alignment.loc["PPG", "max_gap_s"] > MAX_PPG_GAP_S:
    raise ValueError("Split PPG into continuous runs before pulse detection")
display(alignment)
print(f"Common observed support: {common_s / 3600:.2f} h. Whole recording; sleep UNVERIFIED.")
print("No independent HR/PWA clock shift: both are derived from the same optical channel.")

## Movement bursts and strict interburst periods

Detect on the complete accelerometer acquisition so boundary bursts cannot be
lost by first cropping to quiet/sleep intervals. Reuse `detect_bursts_in_intervals`
and its 0.1–10 Hz, order-8 zero-phase preprocessing, envelope, and <5 s merge rule.
Quiet intervals remain separate. Short intervals are retained in the diagnostic
table but contribute no window. There is no concatenation or shortened fallback.

In [ ]:
acc_table = raw_acc.rename(columns={"x": "accel_x", "y": "accel_y", "z": "accel_z"})
acc_table = acc_table.rename_axis("sample_time").reset_index()
acquisition = validate_intervals(pd.DataFrame({"start": [raw_acc.index[0]], "end": [raw_acc.index[-1]]}))
detection = detect_bursts_in_intervals(
    acc_table, acquisition, sampling_rate=ACC_FS, input_unit=ACC_INPUT_UNIT,
    max_gap_s=MAX_ACC_GAP_S, alfa=ACC_THRESHOLD_G, merge_gap_s=5,
)
coverage = intersect_intervals(detection["intervals"], pd.DataFrame({"start": [common_start], "end": [common_end]}))
periods = quiet_periods(detection["bursts"], coverage, CONFIG)
retained = periods.loc[periods.long_enough]
print(f"{len(detection['bursts'])} bursts; {len(periods)} interburst periods after buffers.")
print(f"{len(retained)} quiet periods ≥{CONFIG.window_s / 60:g} min, totaling {retained.duration_s.sum() / 60:.2f} min.")
print(f"{int(periods.geometric_windows.sum())} movement-only complete windows before beat-quality checks.")
display(detection["quality"])
display(periods)

## Beat-to-beat signals and quality

PWA is paired peak minus preceding foot in filtered optical units. Heart period
is **PP**, the difference between consecutive feet, in seconds; HR is `60 / PP`.
No ECG is present, so these are not RR/NN measurements. The first unknown PP is
omitted. Interval values are assigned to their temporal midpoints; PWA keeps its
own peak time. Each series is independently interpolated onto the same grid,
avoiding an artificial relative shift from assigning both to one beat timestamp.

The existing interval-based Lipponen–Tarvainen classifier flags suspected ectopy,
missed/extra beats and long/short intervals without correcting peaks. Morphology,
positive PWA, PP bounds, a 5 s waveform edge guard, finite variance, minimum beat
count, and maximum interpolation gap are also checked. The default excludes any
flagged beat in conservative window support; changing `max_bad_fraction` relaxes
this criterion explicitly. These checks cannot prove absence of ectopy or PWA
artifacts. No rolling average or silent artifact reconstruction is used.

In [ ]:
beats, filtered_ppg, pulse_quality = extract_beats(
    raw_ppg, fs=PPG_FS, config=CONFIG, polarity=PPG_POLARITY,
    max_raw_gap_s=MAX_PPG_GAP_S, edge_guard_s=5,
)
display(pd.Series(pulse_quality, name="Pulse extraction quality"))
display(beats.head())

# A short pulse-fiducial check is additional to the four requested result figures.
preview_start = common_start + pd.Timedelta(seconds=60)
preview_end = preview_start + pd.Timedelta(seconds=15)
preview = beats.loc[(beats.foot_time >= preview_start) & (beats.pwa_time < preview_end)]
fig_fiducials, ax = plt.subplots(figsize=(13, 3))
wave = filtered_ppg.loc[preview_start:preview_end]
ax.plot(wave.index, wave, color=".3", lw=.8, label="Filtered PPG")
ax.scatter(preview.foot_time, filtered_ppg.loc[preview.foot_time], marker="v", label="Paired feet")
ax.scatter(preview.pwa_time, filtered_ppg.loc[preview.pwa_time], marker="^", label="Paired peaks")
ax.set(title="Inspect pulse polarity, onset pairing and amplitude", ylabel="PPG (a.u.)")
ax.legend(ncol=3)
fig_fiducials.tight_layout()
plt.show()

## CCF computation and available data

Each retained quiet period is trimmed to common beat support so interpolation
never extrapolates from a moving or missing edge. Windows start there and repeat
every 60 s; each contains exactly 1,200 samples and is entirely within its quiet
period. Movement-only availability and signal-supported availability are both
reported, so the lost edge support is visible.

For window-demeaned arrays x and y and lag k samples, the implementation is

`CCF(k/fs) = sum(overlapping x[i] * y[i+k]) / sqrt(sum(x**2) * sum(y**2))`.

This uses fixed full-window variance normalization (autocorrelation at zero is 1),
without wrapping, per-lag re-centering, or lag-length inflation. **Positive τ
means PWA follows HR or PP**, exactly `CCF(τ) = corr(X(t), PWA(t+τ))`.
“Unfiltered” means no additional filtering of beat fluctuations; the optical
waveform still receives the pulse-extraction bandpass.

The low-pass comparison applies a 10th-order Butterworth at 0.15 Hz to both
4 Hz signals using SOS forward/reverse filtering, with 30 s odd-reflection padding
from the same window. This avoids phase delay and never filters across quiet
periods. Double passing squares the amplitude response (effective order 20,
−6 dB at the nominal cutoff). Edge estimates remain sensitive to reflection.
Only window means are removed; no linear trend or high-pass component is removed.
The original 2008 text does not specify phase implementation; the authors'
[2017 methods](https://doi.org/10.3389/fphys.2017.00163) explicitly use forward/reverse filtering.

In [ ]:
result = analyze_quiet_ccf(beats, periods, CONFIG)
counts = availability(result)
display(counts)
if counts.eligible_windows < MIN_WINDOWS_TO_SUMMARIZE:
    warnings.warn(f"Only {int(counts.eligible_windows)} eligible complete windows. Insufficient for a stable summary; methodology unchanged.")
if counts.distinct_contributing_quiet_periods < 2:
    warnings.warn("Fewer than two contributing quiet periods: period-level SEM is unavailable.")
print("Quiet periods are distinct nonoverlapping units, not proven biologically independent observations.")
print("One participant/recording: no participant-level inference or statistical tests on overlapping windows.")
if not result["windows"].empty:
    display(result["windows"].groupby(["included", "exclusion_reason"], dropna=False).size().rename("windows"))
display(result["windows"].head())

## Whole-recording overview

Red marks detected bursts, green shows quiet periods at least five minutes long,
and blue marks eligible overlapping windows. Gold highlights the representative
window and up to two additional windows, selected by temporal spacing.

In [ ]:
representative = representative_signals(beats, result, REPRESENTATIVE_WINDOW_ID)
eligible_ids = result["windows"].loc[result["windows"].included, "window_id"].to_numpy()
highlighted = [] if representative is None else [int(representative[0].window_id)]
if len(eligible_ids):
    candidates = eligible_ids[np.linspace(0, len(eligible_ids) - 1, MAX_HIGHLIGHTED_WINDOWS, dtype=int)]
    highlighted = list(dict.fromkeys(highlighted + candidates.tolist()))[:MAX_HIGHLIGHTED_WINDOWS]
fig_overview = plot_overview(detection, result, highlighted)
plt.show()

## Representative complete window

The default choice minimizes bad fraction and interpolation gap, breaking ties by
time, rather than maximizing a CCF feature. Standardization is for plotting only;
unfiltered and low-pass series are each standardized separately. The third panel
retains PWA's original amplitude scale.

In [ ]:
fig_representative = plot_representative(representative)
plt.show()

## Mean CCF and uncertainty

The top row meets the requested mean ± SEM across eligible windows, but its SEM
is **descriptive only** because windows overlap by four minutes. The bottom row
first averages curves within each quiet period and then weights periods equally.
It provides the number of contributing periods and avoids weighting a long quiet
period simply by its larger window count. SEM is missing for one period/window.
Periods can still be dependent within a person. With multiple participants,
average within participant first and compute uncertainty across participants.

The paper's Fig. 2 summarizes scored states and subjects; our panels compare
HR and PP for one recording without asserting those physiological state effects.

In [ ]:
fig_ccf = plot_ccf(result)
plt.show()

# Extrema of aggregated curves differ from averages of window extrema.
aggregate_extrema = []
for keys, frame in result["summary_curves"].groupby(["aggregation", "signal", "mode"]):
    frame = frame.sort_values("lag_s")
    aggregate_extrema.append(dict(zip(["aggregation", "signal", "mode"], keys),
                                  **extrema(frame["mean"].to_numpy(), frame.lag_s.to_numpy())))
aggregate_extrema = pd.DataFrame(aggregate_extrema)
display(aggregate_extrema)

## Window extrema

Show global minima/maxima and their lags across ±25 s. The CSV flags extrema on
the lag boundary; ties use the lowest lag. A global maximum need not be positive,
nor a minimum negative. These distributions mix overlapping observations and
are not used for independent-window hypothesis tests.

In [ ]:
fig_extrema = plot_extrema(result)
plt.show()

## Export local results

All result files are saved in Git-ignored `offline_processing/outputs/`.
`window_results.csv` contains every evaluated window (including exclusions),
timestamps, quiet IDs, HR/PP × unfiltered/low-pass extrema and lags, edge flags,
beat counts, invalid fractions, interpolation gaps, signal variability, and QC.
Separate tables preserve geometric availability, beat timing, full CCF curves,
equal-period aggregation, and aggregate-curve extrema. Configuration and package
versions support reproducibility. Original notebooks and their outputs are untouched.

In [ ]:
run_dir = OUTPUT_ROOT / f"{raw_ppg.index[0]:%Y%m%dT%H%M%S}_whole_unverified"
run_dir.mkdir(parents=True, exist_ok=True)
for name, frame in {
    "window_results": result["windows"], "quiet_periods": result["periods"].reset_index(),
    "movement_bursts": detection["bursts"], "beat_signals": beats,
    "window_ccf_curves": result["curves"], "quiet_period_ccf_curves": result["period_curves"],
    "summary_ccf_curves": result["summary_curves"], "aggregate_curve_extrema": aggregate_extrema,
    "stream_alignment": alignment.reset_index(),
}.items():
    frame.to_csv(run_dir / f"{name}.csv", index=False)
counts.to_csv(run_dir / "availability.csv", header=True)
for name, fig in {"01_overview": fig_overview, "02_representative": fig_representative,
                  "03_ccf": fig_ccf, "04_window_extrema": fig_extrema,
                  "00_fiducials": fig_fiducials}.items():
    fig.savefig(run_dir / f"{name}.png", dpi=160, bbox_inches="tight")
try:
    revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROCESSING, text=True).strip()
except (OSError, subprocess.CalledProcessError):
    revision = None
metadata = {
    "config": asdict(CONFIG), "sleep_status": "whole recording, sleep unverified",
    "clock_zone_declaration": SENSOR_CLOCK_ZONE, "ppg_fs": PPG_FS, "acc_fs": ACC_FS,
    "acc_threshold_g": ACC_THRESHOLD_G, "acc_input_unit": ACC_INPUT_UNIT,
    "max_ppg_gap_s": MAX_PPG_GAP_S, "max_acc_gap_s": MAX_ACC_GAP_S,
    "pulse_extraction": pulse_quality, "git_revision": revision,
    "source_sha256": {name: hashlib.sha256((PROCESSING / name).read_bytes()).hexdigest()
                       for name in ["polar_quiet_ccf.ipynb", "polar_ccf.py", "ppg_fiducials.py",
                                    "polar_interpretability.py", "detect_acc_bursts.py", "gp_hrv_artifacts.py"]},
    "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scipy", "neurokit2", "matplotlib"]},
    "lag_convention": "corr(X(t), PWA(t+tau)); positive tau means PWA follows X",
    "normalization": "full-window demeaning and fixed full-window energy product",
    "inference": "descriptive; overlapping windows are not independent; one participant",
}
(run_dir / "analysis_metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
print(f"Saved results and figures to {run_dir}")

## Interpretation limits

Inspect the pulse pairing and movement threshold before interpreting a curve.
PWA depends on optical contact, vascular tone and the preprocessing response;
PPG PP includes pulse-arrival variation and is not ECG RR. The HR reciprocal
transformation can change CCF shape as well as sign. Quiet wakefulness can occur
throughout this recording. No sleep-stage, causal, pressure, or baroreflex
conclusion follows from these plots.

Zero or few eligible windows are a valid outcome. Review exported exclusions or
collect longer clean quiet periods; do not silently concatenate intervals,
shorten five-minute windows, or treat additional overlapping windows as new subjects.